# Mainly CatBoost

In [1]:
import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error
import warnings

warnings.simplefilter("ignore")

/home/sophia0805/projects/haggle/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("train.csv")

X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"])
y = df["hours_spent"]
y_log = np.log1p(y)

print(f"Rows: {len(df):,}")
print(f"Features: {X.shape[1]}")

Rows: 4,500
Features: 69


In [3]:
PRIOR_MAP = {"0": 0, "1-2": 1, "3-5": 2, "6+": 3}
AGE_MAP = {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3}
BOOL_MAP = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}

CAT_COLS = [
    "primary_language",
    "framework",
    "hosting_type",
    "commit_cadence",
    "project_category",
]

BOOL_COLS = [
    "uses_game_engine", "has_tests", "has_ci_config", "has_dockerfile",
    "has_backend", "has_database", "has_custom_art",
    "readme_has_install_instructions", "has_license", "has_docs_folder",
    "url_is_live", "is_mobile_responsive", "has_custom_domain",
    "is_multiplayer", "mentions_ai_api", "is_clone_or_tutorial",
    "is_forked_from_template",
]


def clean_features(X):
    X = X.copy()
    X["author_prior_projects"] = X["author_prior_projects"].astype(str).map(PRIOR_MAP)
    X["author_account_age_years"] = X["author_account_age_years"].astype(str).map(AGE_MAP)

    for column in BOOL_COLS:
        if column in X.columns:
            X[column] = X[column].map(BOOL_MAP)

    for column in CAT_COLS:
        X[column] = X[column].fillna("missing").astype(str)

    return X


def divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


def add_features(X):
    X = clean_features(X)
    X = X.copy()

    X["lines_per_active_day"] = divide(X.lines_added_total, X.active_commit_days)
    X["loc_per_commit"] = divide(X.lines_of_code, X.num_commits)
    X["commits_per_active_day"] = divide(X.num_commits, X.active_commit_days)
    X["active_frac"] = divide(X.active_commit_days, X.project_span_days + 1)
    X["total_lines_touched"] = np.log1p(X.lines_added_total + X.lines_deleted_total)
    X["biggest_commit_share"] = divide(X.max_lines_in_one_commit, X.lines_added_total)
    X["n_assets"] = X[[
        "num_image_files", "num_audio_files", "num_3d_model_files", "num_font_files"
    ]].sum(axis=1, min_count=1)
    X["readme_richness"] = X[[
        "readme_num_headings", "readme_num_images", "readme_num_code_blocks"
    ]].sum(axis=1, min_count=1)
    X["loc_x_days"] = np.log1p(X.lines_of_code) * np.log1p(X.active_commit_days)
    X["loc_template_adj"] = (
        np.log1p(X.lines_of_code)
        * (1 - 0.5 * X.is_forked_from_template.fillna(0))
    )
    X["loc_template_adj"] = np.log1p(X.lines_of_code) * (1 - 0.5 * X.is_forked_from_template.fillna(0))
    X["log_code_velocity"] = np.log1p(divide(X["lines_added_total"], X["active_commit_days"]))
    X["loc_per_file"] = np.log1p(divide(X["lines_of_code"], X["num_source_files"]))
    X["density_active_days"] = divide(X["active_commit_days"], X["project_span_days"] + 1)
    X["log_commits_intensity"] = np.log1p(divide(X["num_commits"], X["project_span_days"] + 1))
    X["non_code_file_ratio"] = divide(X["n_assets"], X["num_source_files"] + X["n_assets"] + 1)
    return X

Xf = add_features(X)

In [4]:
MODEL_PARAMS = {
    "cat_features": CAT_COLS,
    "iterations": 1245,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 3,
    "rsm": 0.5,
    "loss_function": "RMSE",
    "random_seed": 42,
    "allow_writing_files": False,
    "verbose": 0,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(Xf))
fold_scores = []

for train_idx, val_idx in cv.split(Xf, df["hours_bucket"]):
    model = CatBoostRegressor(**MODEL_PARAMS)
    model.fit(Xf.iloc[train_idx], y_log.iloc[train_idx])

    oof_predictions[val_idx] = model.predict(Xf.iloc[val_idx])
    score = root_mean_squared_error(y_log.iloc[val_idx], oof_predictions[val_idx])
    fold_scores.append(score)

print("Fold scores:", ", ".join(f"{score:.5f}" for score in fold_scores))
print(f"Mean CV score: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print(f"OOF score: {root_mean_squared_error(y_log, oof_predictions):.5f}")

Fold scores: 0.64999, 0.68289, 0.65303, 0.68005, 0.65295
Mean CV score: 0.66378 +/- 0.01451
OOF score: 0.66394


In [5]:
final_model = CatBoostRegressor(**MODEL_PARAMS)
final_model.fit(Xf, y_log)
final_model.save_model("model.cbm")

importance = final_model.get_feature_importance(prettified=True)
print(importance.head(25).to_string(index=False))

              Feature Id  Importances
              loc_x_days     6.270440
  description_word_count     6.094178
      active_commit_days     3.764210
 median_lines_per_commit     3.354143
        loc_template_adj     2.823465
            hosting_type     2.733206
        project_category     2.715694
        primary_language     2.550745
 pct_generic_commit_msgs     2.464346
        num_source_files     2.308038
    biggest_commit_share     2.270422
pct_code_in_first_commit     2.215608
     pct_commits_weekend     2.211395
             num_commits     2.129877
       lines_added_total     2.064550
       readme_word_count     2.043790
   avg_commit_msg_length     2.028171
    pct_primary_language     2.007014
      comment_line_ratio     1.805493
  commits_per_active_day     1.710941
     total_lines_touched     1.704027
  pct_commits_late_night     1.637778
       project_span_days     1.520265
    lines_per_active_day     1.518425
          loc_per_commit     1.513984


In [6]:
residuals = oof_predictions - y_log
print(pd.DataFrame({
    "hours_bucket": df["hours_bucket"],
    "residual_log_hours": residuals,
}).groupby("hours_bucket")["residual_log_hours"].agg(["mean", "std", "count"]))

                  mean       std  count
hours_bucket                           
10-30        -0.299997  0.506263   1120
100+         -1.673278  0.593439     63
30-100       -0.890537  0.558572    332
<10           0.236682  0.538118   2985


# Making Ensemble Predictions
Adding LightGBM, XGBoost, and Ridge Regression (hopefully it works)

In [9]:
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
X_lgb = Xf.copy()
for col in CAT_COLS:
    X_lgb[col] = X_lgb[col].astype("category")

X_ohe = pd.get_dummies(Xf, columns=CAT_COLS, drop_first=True, dtype=float)   # NaN kept for XGBoost

num_cols = Xf.columns.difference(CAT_COLS)
skewed = [c for c in num_cols if Xf[c].min() >= 0 and Xf[c].max() > 50]
X_lin = X_ohe.copy()
X_lin[skewed] = np.log1p(X_lin[skewed])
miss = Xf[num_cols].isna()
miss = miss.loc[:, miss.any()].astype(float).add_suffix("_missing")
X_lin = pd.concat([X_lin, miss], axis=1).fillna(0)

LGB_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbose": -1,
    "random_state": 42,
}
LGB_ROUNDS = 1000

XGB_PARAMS = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "random_state": 42,
    "n_jobs": -1,
}

RIDGE_ALPHA = 10.0

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(Xf))
oof_xgb = np.zeros(len(Xf))
oof_ridge = np.zeros(len(Xf))
lgb_scores, xgb_scores, ridge_scores = [], [], []

for train_idx, val_idx in cv.split(Xf, df["hours_bucket"]):
    y_tr, y_va = y_log.iloc[train_idx], y_log.iloc[val_idx]

    # LightGBM
    lgb_model = lgb.train(LGB_PARAMS, lgb.Dataset(X_lgb.iloc[train_idx], y_tr), num_boost_round=LGB_ROUNDS)
    oof_lgb[val_idx] = lgb_model.predict(X_lgb.iloc[val_idx])
    lgb_scores.append(root_mean_squared_error(y_va, oof_lgb[val_idx]))

    # XGBoost
    xgb_model = XGBRegressor(**XGB_PARAMS)
    xgb_model.fit(X_ohe.iloc[train_idx], y_tr, verbose=False)
    oof_xgb[val_idx] = xgb_model.predict(X_ohe.iloc[val_idx])
    xgb_scores.append(root_mean_squared_error(y_va, oof_xgb[val_idx]))

    # Ridge
    ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
    ridge_model.fit(X_lin.iloc[train_idx], y_tr)
    oof_ridge[val_idx] = ridge_model.predict(X_lin.iloc[val_idx])
    ridge_scores.append(root_mean_squared_error(y_va, oof_ridge[val_idx]))

for name, scores, oof in [
    ("LightGBM", lgb_scores, oof_lgb),
    ("XGBoost", xgb_scores, oof_xgb),
    ("Ridge", ridge_scores, oof_ridge),
]:
    print(f"--- {name} ---")
    print("Fold scores:", ", ".join(f"{score:.5f}" for score in scores))
    print(f"Mean CV score: {np.mean(scores):.5f} +/- {np.std(scores):.5f}")
    print(f"OOF score: {root_mean_squared_error(y_log, oof):.5f}")

In [ ]:
from scipy.optimize import minimize

def objective(weights):
    w1, w2, w3, w4 = weights
    blend = w1 * oof_predictions + w2 * oof_lgb + w3 * oof_xgb + w4 * oof_ridge
    return root_mean_squared_error(y_log, blend)

constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1)] * 4

res = minimize(objective, [0.25, 0.25, 0.25, 0.25], method='SLSQP', bounds=bounds, constraints=constraints)

w1, w2, w3, w4 = res.x
print("--- Optimized Ensemble Weights ---")
print(f"CatBoost Weight: {w1:.3f}")
print(f"LightGBM Weight: {w2:.3f}")
print(f"XGBoost Weight:  {w3:.3f}")
print(f"Ridge Weight:    {w4:.3f}")
print(f"\nOptimized 4-Model Ensemble OOF RMSE Score: {res.fun:.5f}")